# SPACEc tissue-context and cellular-neighborhood interaction analysis

This notebook contains the study-specific SPACEc code used to:

1. construct tissue-context maps from pre-assigned cellular neighborhoods;
2. visualize a representative tissue-context / barycentric map; and
3. quantify pairwise cellular-neighborhood interactions across tissue samples.

The input is a processed AnnData object containing `sample` and `cn` in `adata.obs`
and spatial coordinates in `adata.obsm["spatial"]`.

Set `DATA_PATH` to the location of the processed AnnData file before running.


In [ ]:
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import spacec as sp

from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests


## Load processed spatial-proteomic data

A repository-relative placeholder is used here rather than a local-computer path.


In [ ]:
DATA_PATH = Path("data/adata.h5ad")

adata = sc.read_h5ad(
    DATA_PATH,
    backed="r"
)

print(adata)


## Define tissue-domain labels and spatial coordinates


In [ ]:
# SPACEc requires explicit x/y coordinate columns.
coordinates = np.asarray(
    adata.obsm["spatial"][:, :2],
    dtype=np.float32
)

adata.obs["x"] = coordinates[:, 0]
adata.obs["y"] = coordinates[:, 1]

del coordinates

# Map the original cellular-neighborhood labels to the tissue-domain labels
# used in the manuscript.
cn_to_region = {
    "region_1": "MY-2",
    "region_2": "AL",
    "region_3": "AL",
    "region_4": "MY-1",
    "region_5": "SI",
    "region_6": "LA",
}

adata.obs["region_type"] = (
    adata.obs["cn"]
    .map(cn_to_region)
    .astype("category")
)

region_palette = {
    "AL": "#9669BA",
    "SI": "#BABD3A",
    "MY-2": "#FEA1A1",
    "MY-1": "#F26D6C",
    "LA": "#69A7CB",
}


## Build the SPACEc tissue-context map

The same SPACEc parameters used for the manuscript analysis are retained here.
The resulting `w` table stores the local tissue-context combinations used below
for pairwise interaction quantification.


In [ ]:
cn_map_all = sp.tl.build_cn_map(
    adata=adata,
    cn_col="region_type",
    unique_region="sample",
    k=70,
    X="x",
    Y="y",
    threshold=0.85,
    per_keep_thres=0.85,
)


## Pairwise cellular-neighborhood interaction quantification

For each tissue sample, an interaction pair is counted when both domains occur
within the same retained SPACEc local context. A context containing three domains
therefore contributes once to each of the three corresponding domain pairs.

Interaction frequency is reported as the percentage of retained local contexts
containing each pair.


In [ ]:
sample_col = "sample"

w = cn_map_all["w"].copy()

# SPACEc stores the retained domains for each local context in "combination".
w["combination_set"] = w["combination"].apply(set)

pairs = {
    "MY-1-SI": {"MY-1", "SI"},
    "LA-SI": {"LA", "SI"},
    "MY-1-LA": {"MY-1", "LA"},
}

results = []

for sample, df_sample in w.groupby(sample_col):
    n_total = len(df_sample)

    for pair_name, pair_set in pairs.items():
        contains_pair = df_sample["combination_set"].apply(
            lambda domains: pair_set.issubset(domains)
        )

        n_pair = int(contains_pair.sum())
        percentage = (n_pair / n_total * 100) if n_total > 0 else np.nan

        results.append(
            {
                "sample": sample,
                "interaction": pair_name,
                "n_pair_windows": n_pair,
                "n_total_windows": n_total,
                "percentage": percentage,
            }
        )

cn_interaction_df = pd.DataFrame(results)

cn_interaction_df.head()


## Paired statistical comparison across tissue samples

The three pairwise interfaces are compared using two-sided paired Wilcoxon
signed-rank tests, followed by Benjamini–Hochberg correction across the three
comparisons.


In [ ]:
cn_interaction_wide = cn_interaction_df.pivot(
    index="sample",
    columns="interaction",
    values="percentage",
)

comparisons = [
    ("MY-1-SI", "MY-1-LA"),
    ("LA-SI", "LA-LA"),
    ("MY-1-LA", "LA-SI"),
]

stats_results = []

for group1, group2 in comparisons:
    paired = cn_interaction_wide[[group1, group2]].dropna()

    statistic, p_value = wilcoxon(
        paired[group1],
        paired[group2],
        alternative="two-sided",
    )

    stats_results.append(
        {
            "group1": group1,
            "group2": group2,
            "n_samples": len(paired),
            "median_group1": paired[group1].median(),
            "median_group2": paired[group2].median(),
            "statistic": statistic,
            "pvalue": p_value,
        }
    )

stats_df = pd.DataFrame(stats_results)

stats_df["padj"] = multipletests(
    stats_df["pvalue"],
    method="fdr_bh",
)[1]

stats_df


## Representative tissue-context visualization

The representative sample used in the original notebook is retained here.
Only metadata and spatial coordinates are copied because the SPACEc context-map
visualization does not require the expression matrix.


In [ ]:
REPRESENTATIVE_SAMPLE = "P7"

sample_mask = (
    adata.obs["sample"].astype(str) == REPRESENTATIVE_SAMPLE
).to_numpy()

sample_positions = np.flatnonzero(sample_mask)

if len(sample_positions) == 0:
    raise ValueError(
        f"{REPRESENTATIVE_SAMPLE!r} was not found in adata.obs['sample']."
    )

obs_rep = adata.obs.iloc[sample_positions].copy()
spatial_rep = np.asarray(
    adata.obsm["spatial"][sample_positions, :2],
    dtype=np.float32,
)

for column in ["sample", "cn", "region_type"]:
    if column in obs_rep.columns:
        obs_rep[column] = (
            obs_rep[column]
            .astype("category")
            .cat.remove_unused_categories()
        )

adata_rep = ad.AnnData(obs=obs_rep)
adata_rep.obsm["spatial"] = spatial_rep


In [ ]:
cn_map_rep = sp.tl.build_cn_map(
    adata=adata_rep,
    cn_col="region_type",
    unique_region="sample",
    k=70,
    X="x",
    Y="y",
    threshold=0.85,
    per_keep_thres=0.85,
)

sp.pl.cn_map(
    cnmap_dict=cn_map_rep,
    adata=adata_rep,
    cn_col="region_type",
    palette=region_palette,
    figsize=(25, 20),
    savefig=False,
    output_fname="",
)


## Barycentric representation of local tissue context

This visualization represents the relative contributions of the
granuloma-associated, stromal and lymphoid domains within local SPACEc contexts.


In [ ]:
sp.pl.BC_projection(
    adata=adata_rep,
    cnmap_dict=cn_map_rep,
    cn_col="region_type",
    plot_list=["granuloma_2", "stromal", "tls"],
    cn_col_annt="region_type",
    palette=region_palette,
    figsize=(5, 5),
    rand_seed=1,
    n_num=None,
    threshold=0.69,
)
